# PawNote — Day 3~5 Colab

> **2026-09-07 — ①~⑦ 실행 완료.** 베이스라인 2 40.5% · 파인튜닝 90.0%.
> **다음은 ⑧⑨ GGUF 변환**이다. 세션이 새로 시작됐으면 ①②③ 만 다시 돌리고 ⑧ 로 간다.

**로컬에서 뽑아둔 프롬프트를 읽어 실행만 한다.** 프로젝트 코드를 여기서 clone 하지 않는다 —
프롬프트를 Colab에서 다시 조립하면 환경에 따라 달라질 수 있고, 그러면 GPT-4.1과
**같은 시험을 본 게 아니게 된다.**

드라이브 `MyDrive/PawNote/` 에 미리 올려둘 것

| 파일 | 쓰임 |
| --- | --- |
| `prompts_base.jsonl` | 베이스라인 2 입력 — 툴 스펙 + few-shot 8쌍 + 발화 |
| `prompts_ft.jsonl` | 파인튜닝 모델 입력 — **발화 한 줄뿐** (결정 2) |
| `sft_train.jsonl` · `sft_val.jsonl` | 학습 1,960 / 검증 200 |

**채점은 로컬에서 한다.** 이 노트북은 `pred_*.jsonl` 만 만든다. 채점기를 두 군데서
돌리면 언젠가 한쪽만 고쳐진다.

순서 — ① 설치 ② 마운트 ③ **dtype 확인** ④ 베이스라인 2 ⑤ 학습 ⑥ 저장 ⑦ ft 예측


In [ ]:
# ① 설치 — 5분쯤 걸린다
%pip install -q unsloth

# 위가 실패하면 나이틀리로 (Colab 런타임과 버전이 어긋날 때)
# %pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git


In [ ]:
# ② 드라이브 마운트
#
# 체크포인트를 로컬 디스크에 두면 세션이 끊길 때 통째로 사라진다.
import json, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJ = Path('/content/drive/MyDrive/PawNote')

NEED = ['prompts_base.jsonl', 'prompts_ft.jsonl', 'sft_train.jsonl', 'sft_val.jsonl']
missing = [f for f in NEED if not (PROJ / f).exists()]
assert not missing, f'드라이브에 없다 — {missing}'

for f in NEED:
    n = sum(1 for _ in open(PROJ / f, encoding='utf-8'))
    print(f'{f:22s} {n:5d}건')


In [ ]:
# ③ ⚠ 최우선 — dtype 확인. 여기서 막히면 그날 GPU 할당량이 통째로 날아간다
#
# bnb-4bit 설정의 compute dtype 기본값이 bfloat16 인데 T4(Turing)는 bf16 미지원이다.
# 명시하지 않으면 **로드는 되고 학습에서 터진다** — 가장 비싼 실패 방식이다.
import torch
from unsloth import FastLanguageModel

print(torch.cuda.get_device_name(0))
BF16 = torch.cuda.is_bf16_supported()
print('bf16 지원', BF16)

MODEL = 'unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit'
MAXLEN = 4096   # 베이스라인 2 프롬프트가 툴 스펙 + few-shot 8쌍이라 길다


def load_base():
    m, t = FastLanguageModel.from_pretrained(
        model_name=MODEL,
        max_seq_length=MAXLEN,
        dtype=torch.bfloat16 if BF16 else torch.float16,
        load_in_4bit=True,
    )
    qc = getattr(m.config, 'quantization_config', None)
    cd = str(getattr(qc, 'bnb_4bit_compute_dtype', qc))
    print('compute dtype —', cd)
    assert BF16 or 'bfloat16' not in cd, 'T4인데 compute dtype이 bf16이다. 여기서 멈춘다'
    return m, t


model, tokenizer = load_base()


In [ ]:
# ④ 베이스라인 2 — base 모델 + 툴 스펙 + few-shot 8쌍
#
# **파인튜닝 기여도 증명의 기준선이다. 이게 없으면 프로젝트가 성립하지 않는다.**
# LoRA를 붙이기 **전에** 돌린다. 순서가 바뀌면 기준선이 오염된다.
#
# 평가에서 다양성은 해악이므로 샘플링을 끈다. 로컬의 temperature=0 과 같은 자리다.
#
# 경고를 끄는 이유 — Qwen3 기본 설정의 max_length 가 262144 라서 우리가 준
# max_new_tokens=160 과 둘 다 설정됐다는 경고가 **건마다** 뜬다. 동작은 의도대로지만
# 200줄이 찍혀 진짜 결과를 덮는다. 라벨이 106자짜리 JSON 한 줄이라 160이면 넉넉하고,
# 상한을 안 걸면 모델이 헛돌 때 한 건에 몇 분씩 잡아먹는다.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

FastLanguageModel.for_inference(model)


def run(prompt_path, out_path, max_new_tokens=160):
    rows = [json.loads(l) for l in open(prompt_path, encoding='utf-8') if l.strip()]
    results, lat = [], []

    for i, r in enumerate(rows):
        text = tokenizer.apply_chat_template(
            r['messages'], tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors='pt').to('cuda')
        n_in = enc['input_ids'].shape[1]

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        ms = (time.perf_counter() - t0) * 1000

        gen = tokenizer.decode(out[0][n_in:], skip_special_tokens=True)
        lat.append(ms)
        results.append({'id': r['id'], 'output': gen.strip(),
                        'latency_ms': round(ms, 1),
                        'input_tokens': int(n_in),
                        'output_tokens': int(out.shape[1] - n_in)})
        if (i + 1) % 25 == 0:
            print(f'  {i + 1}/{len(rows)}', flush=True)

    with open(out_path, 'w', encoding='utf-8') as f:
        for x in results:
            f.write(json.dumps(x, ensure_ascii=False) + '\n')

    s = sorted(lat)
    p = lambda q: s[min(int(len(s) * q), len(s) - 1)]
    print(f'\n{out_path.name} · {len(results)}건')
    print(f'지연 p50 {p(0.5):.0f}ms · p95 {p(0.95):.0f}ms  (T4 숫자다. 서빙 지연은 Day 5)')
    print(f"입력 토큰 건당 {sum(x['input_tokens'] for x in results) / len(results):.0f}")
    return results


base_pred = run(PROJ / 'prompts_base.jsonl', PROJ / 'pred_base.jsonl')
print('\n첫 출력 —', base_pred[0]['output'])


In [ ]:
# ⑤ QLoRA 학습
#
# 모델을 **다시 읽는다.** 위에서 추론 모드로 바꾼 객체에 LoRA를 붙이면 상태가 섞인다.
# 1분이 아까워서 디버깅에 한 시간을 쓸 이유가 없다.
import gc, inspect
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

del model
gc.collect(); torch.cuda.empty_cache()
model, tokenizer = load_base()

LR = 2e-4        # Day 4에서 2~3개만 바꿔 재학습한다. 광범위 탐색은 GPU를 태운다
EPOCHS = 2
TAG = f'lr{LR:g}_ep{EPOCHS}'

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=0,
)

ds = load_dataset('json', data_files={'train': str(PROJ / 'sft_train.jsonl'),
                                      'val': str(PROJ / 'sft_val.jsonl')})
ds = ds.map(lambda b: {'text': [tokenizer.apply_chat_template(m, tokenize=False)
                                for m in b['messages']]}, batched=True)
print(ds)
print(ds['train'][0]['text'])


In [ ]:
# TRL 버전마다 인자 이름이 다르다. 여기서 한 번 맞춰두면 TypeError 로 세션을 안 태운다
_cfg = set(inspect.signature(SFTConfig.__init__).parameters)
_trn = set(inspect.signature(SFTTrainer.__init__).parameters)
LEN_KEY = 'max_seq_length' if 'max_seq_length' in _cfg else 'max_length'
EVAL_KEY = 'eval_strategy' if 'eval_strategy' in _cfg else 'evaluation_strategy'
TOK_KEY = 'tokenizer' if 'tokenizer' in _trn else 'processing_class'
print(LEN_KEY, EVAL_KEY, TOK_KEY)

cfg = SFTConfig(**{
    'dataset_text_field': 'text',
    LEN_KEY: 512,          # 발화 + 라벨이라 짧다. 4096을 쓰면 메모리만 먹는다
    EVAL_KEY: 'epoch',
    'per_device_train_batch_size': 8,
    'gradient_accumulation_steps': 2,     # 실효 배치 16
    'num_train_epochs': EPOCHS,
    'learning_rate': LR,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'linear',
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'logging_steps': 10,
    'fp16': not BF16, 'bf16': BF16,
    'seed': 0,
    'output_dir': '/content/outputs',
    'report_to': 'none',
})

trainer = SFTTrainer(model=model, train_dataset=ds['train'], eval_dataset=ds['val'],
                     args=cfg, **{TOK_KEY: tokenizer})

# **라벨(assistant)에만 손실을 건다.** 발화까지 외우게 하면 형식 안정성이 덜 오른다.
# 이 프로젝트가 기대하는 하이라이트가 형식 안정성이므로 그냥 넘길 자리가 아니다.
trainer = train_on_responses_only(trainer,
                                  instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')

stats = trainer.train()
print(stats)


In [ ]:
# ⑥ 체크포인트를 드라이브에 저장 — 학습 직후 **바로** 한다
save_to = PROJ / f'adapter_{TAG}'
model.save_pretrained(str(save_to))
tokenizer.save_pretrained(str(save_to))
print('저장', save_to)

hist = [h for h in trainer.state.log_history if 'loss' in h or 'eval_loss' in h]
(PROJ / f'loss_{TAG}.json').write_text(
    json.dumps(hist, ensure_ascii=False, indent=1), encoding='utf-8')

import matplotlib.pyplot as plt
tr = [(h['step'], h['loss']) for h in hist if 'loss' in h]
ev = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]
plt.plot(*zip(*tr), label='train')
if ev:
    plt.plot(*zip(*ev), 'o-', label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3); plt.show()


In [ ]:
# ⑦ 파인튜닝 모델 예측 — 입력은 발화 한 줄뿐이다 (결정 2)
#
# 여기서 나오는 입력 토큰 수가 베이스라인 2의 몇 분의 1인지가 결정 2의 성과다.
FastLanguageModel.for_inference(model)
ft_pred = run(PROJ / 'prompts_ft.jsonl', PROJ / f'pred_ft_{TAG}.jsonl')
print('\n첫 출력 —', ft_pred[0]['output'])


---

# Day 5 — GGUF 변환 (여기부터는 학습이 끝난 뒤)

세션이 끊겼다면 **①②③만 다시 돌리고** 아래로 온다. 학습은 다시 안 해도 된다 —
어댑터가 드라이브에 있다.


In [ ]:
# ⑧ 어댑터를 읽는다 — 세션이 새로 시작됐을 때만 필요하다
#
# ⑤~⑦ 을 방금 돌렸다면 이 셀은 건너뛴다. `model` 이 이미 학습된 상태다.
TAG = 'lr0.0002_ep2'
ADAPTER = PROJ / f'adapter_{TAG}'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER),
    max_seq_length=MAXLEN,
    dtype=torch.bfloat16 if BF16 else torch.float16,
    load_in_4bit=True,
)
print('어댑터 로드', ADAPTER)


In [ ]:
# ⑨ GGUF 변환 — 어댑터를 base 에 병합하고 Ollama 가 읽는 한 파일로 만든다
#
# q4_k_m 은 4비트 양자화(가중치를 4비트로 줄여 파일과 메모리를 아끼는 것) 중
# 품질/크기 균형이 가장 무난한 방식이다. **온디바이스 타당성의 근거가 이 파일 크기**이므로
# 변환 후 크기를 반드시 적어둔다.
#
# 20~40분 걸린다. 변환 도구(llama.cpp)를 처음 받으면 더 걸린다.
GGUF_DIR = PROJ / f'gguf_{TAG}'
model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method='q4_k_m')

for f in sorted(GGUF_DIR.glob('*.gguf')):
    print(f'{f.name}  {f.stat().st_size / 1024**3:.2f} GB')


## 로컬에서 Ollama 서빙 — 여기부터 GPU가 필요 없다

드라이브 `gguf_lr0.0002_ep2/` 에서 `.gguf` 파일을 내려받아 프로젝트 폴더에 둔다.

**① Ollama 설치** — [ollama.com/download](https://ollama.com/download) 에서 받아 설치한다.

**② `Modelfile` 작성** (내려받은 파일 이름으로 바꿀 것)

```
FROM ./unsloth.Q4_K_M.gguf
PARAMETER temperature 0
PARAMETER num_predict 160
```

**③ 등록하고 서빙**

```
ollama create pawnote -f Modelfile
ollama run pawnote "콩이 아침에 사료 반 그릇 먹었어"
```

**④ 200건 재평가** — 변환에서 성능이 떨어지지 않았는지 본다. **90.0%가 유지돼야 한다.**

```
python src/predict.py --target ft --input data/test.jsonl \
    --engine ollama --model pawnote --out data/pred_ollama.jsonl
python src/evaluate.py --gold data/test.jsonl --pred data/pred_ollama.jsonl \
    --details data/eval_ollama.jsonl
```

⚠ **`<think>` 블록이 Colab과 같은 모양인지 확인한다.** 블록 안에 중괄호가 들어가면
채점 규칙상 파싱 실패가 된다. Colab 실측은 200건 전부 빈 블록이었다.

**⑤ CPU-only 지연** — `OLLAMA_NUM_GPU=0` 으로 서빙하고 같은 명령을 다시 돌린다.
온디바이스 타당성은 **GGUF 파일 크기 + CPU 추론 지연**으로 간접 논증한다 (실기기 벤치마크는 범위 밖).

---

## 끝나면 로컬에서 채점한다

드라이브에서 `pred_base.jsonl` · `pred_ft_*.jsonl` 을 내려받아 `data/` 에 두고

```
python src/evaluate.py --gold data/test.jsonl --pred data/pred_base.jsonl   --details data/eval_base.jsonl
python src/evaluate.py --gold data/test.jsonl --pred data/pred_ft_lr0.0002_ep2.jsonl --details data/eval_ft.jsonl
```

**T4 지연은 서빙 숫자가 아니다.** P2 판정에 쓰는 것은 Day 5의 Ollama·CPU-only 측정이다.
여기 숫자는 학습 중 참고용이다.

### 세션이 끊겼다면

③부터 다시 돌린다. ⑥에서 어댑터를 드라이브에 저장했다면 학습은 다시 안 해도 된다 —
`FastLanguageModel.from_pretrained(str(PROJ / 'adapter_...'))` 로 읽으면 ⑦만 다시 돌리면 된다.
